# 02 - Extract Cal-Adapt Analytics Engine holdings (s3://cadcat)

LOCA2-Hybrid 3 km (**California only - the NV side of the basin is a documented gap**,
see `docs/METHODS.md`) and WRF dynamical downscaling for wind. Anonymous Zarr on AWS
Open Data; xarray + s3fs reads pull only the bbox subset - no bulk download.

Requires `s3fs` + `zarr` (see `environment.md`). Every cell degrades to a logged
warning without them.

In [ ]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("02_extract_caladapt")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

In [ ]:
try:
    import s3fs
    import zarr  # noqa: F401
    FS = s3fs.S3FileSystem(anon=True)
    HAS_S3 = True
except ImportError:
    FS = None
    HAS_S3 = False
    log.warning("s3fs/zarr not installed - notebook will only log what it would do. "
                "conda install -n arcgispro-py3 -c conda-forge s3fs zarr")

CAD = cfg["sources"]["caladapt"]
BUCKET = CAD["s3_bucket"]

if HAS_S3:
    top = FS.ls(BUCKET)
    log.info(f"s3://{BUCKET} top level: {top}")

## Explore the catalog
The cadcat layout nests `<activity>/<institution>/<model>/<scenario>/<member>/<table>/<variable>/<grid>/` Zarr stores. This cell walks the LOCA2-Hybrid and WRF trees for the configured models/variables and records what exists - fill `STORES` below from its output if names differ.

In [ ]:
found = []
if HAS_S3:
    for prefix in ["loca2", "wrf"]:
        try:
            for lvl1 in FS.ls(f"{BUCKET}/{prefix}"):
                for lvl2 in FS.ls(lvl1):
                    found.append(lvl2)
        except FileNotFoundError:
            log.warning(f"no s3://{BUCKET}/{prefix} prefix")
    log.info(f"{len(found)} model paths; sample: {found[:8]}")
    pd.Series(found).to_csv(OUTPUTS / "cadcat_paths.csv", index=False)

## Subset LOCA2-Hybrid 3 km + WRF wind
Open each store lazily, slice the bbox, save to `data/raw/caladapt/`. WRF grids are
curvilinear (2-D lat/lon) - subsetting uses a where-mask instead of `.sel` slices.

In [ ]:
def bbox_subset_any(ds):
    """Slice rectilinear lat/lon, or mask curvilinear 2-D coordinates."""
    lat_name = "lat" if "lat" in ds.coords else "latitude"
    lon_name = "lon" if "lon" in ds.coords else "longitude"
    lat, lon = ds[lat_name], ds[lon_name]
    lo_min, lo_max = BBOX["lon_min"], BBOX["lon_max"]
    if float(lon.max()) > 180:
        lo_min, lo_max = lo_min + 360, lo_max + 360
    if lat.ndim == 1:
        lat_asc = bool(lat[0] < lat[-1])
        return ds.sel({lon_name: slice(lo_min, lo_max),
                       lat_name: slice(BBOX["lat_min"], BBOX["lat_max"]) if lat_asc
                       else slice(BBOX["lat_max"], BBOX["lat_min"])})
    mask = ((lat >= BBOX["lat_min"]) & (lat <= BBOX["lat_max"])
            & (lon >= lo_min) & (lon <= lo_max))
    return ds.where(mask, drop=True)


cad_dir = RAW / "caladapt"
cad_dir.mkdir(exist_ok=True)

if HAS_S3:
    import fnmatch
    member_of = cfg["sources"]["loca2"].get("member_overrides", {})
    default_member = cfg["sources"]["loca2"]["member"]
    scenarios = ["historical"] + cfg["sources"]["loca2"]["scenarios"]

    # --- LOCA2-Hybrid 3 km ---
    if CAD["loca2_hybrid_3km"]["enabled"]:
        for gcm in cfg["sources"]["loca2"]["gcms"]:
            hits = fnmatch.filter(found, f"*loca2*/{gcm}*") or \
                   fnmatch.filter(found, f"*loca2*/{gcm.lower()}*")
            if not hits:
                log.warning(f"[3km] no cadcat path for {gcm}")
                continue
            base = hits[0]
            member = member_of.get(gcm, default_member)
            for scen in scenarios:
                for var in CAD["loca2_hybrid_3km"]["variables"]:
                    store = f"{base}/{scen}/{member}/day/{var}"
                    matches = [p for p in FS.glob(store + "*") if FS.exists(p)]
                    if not matches:
                        log.warning(f"[3km] missing store: s3://{store}")
                        continue
                    out = cad_dir / f"loca2hybrid3km_{gcm}_{scen}_{var}__tahoe.nc"
                    if out.exists() and not cfg["run"]["overwrite_downloads"]:
                        continue
                    try:
                        ds = xr.open_zarr(FS.get_mapper(matches[0]), consolidated=True)
                        sub = bbox_subset_any(ds).load()
                        sub.to_netcdf(out)
                        append_manifest({"file": str(out.relative_to(ROOT)),
                                         "source_url": f"s3://{matches[0]}",
                                         "size_bytes": out.stat().st_size,
                                         "sha256": sha256_file(out),
                                         "retrieved_date": str(pd.Timestamp.today().date()),
                                         "notebook": "02_extract_caladapt"})
                        log.info(f"[3km] {out.name} ({out.stat().st_size/1e6:.1f} MB)")
                    except Exception as e:
                        log.warning(f"[3km] FAILED {store}: {e}")

    # --- WRF wind ---
    if CAD["wrf_wind"]["enabled"]:
        wrf_paths = [p for p in found if "/wrf/" in p or p.startswith(f"{BUCKET}/wrf")]
        log.info(f"[wrf] model paths available: {wrf_paths}")
        for base in wrf_paths:
            for scen in scenarios:
                for var in CAD["wrf_wind"]["variables"]:
                    cands = FS.glob(f"{base}/{scen}/*/day/{var}*") + \
                            FS.glob(f"{base}/{scen}/*/1day/{var}*")
                    if not cands:
                        continue
                    name = base.rsplit("/", 1)[1]
                    out = cad_dir / f"wrf_{name}_{scen}_{var}__tahoe.nc"
                    if out.exists() and not cfg["run"]["overwrite_downloads"]:
                        continue
                    try:
                        ds = xr.open_zarr(FS.get_mapper(cands[0]), consolidated=True)
                        sub = bbox_subset_any(ds).load()
                        sub.to_netcdf(out)
                        append_manifest({"file": str(out.relative_to(ROOT)),
                                         "source_url": f"s3://{cands[0]}",
                                         "size_bytes": out.stat().st_size,
                                         "sha256": sha256_file(out),
                                         "retrieved_date": str(pd.Timestamp.today().date()),
                                         "notebook": "02_extract_caladapt"})
                        log.info(f"[wrf] {out.name} ({out.stat().st_size/1e6:.1f} MB)")
                    except Exception as e:
                        log.warning(f"[wrf] FAILED {cands[0]}: {e}")

log.info("Cal-Adapt extract pass complete")